In [ ]:
import pandas as pd
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    import umap.umap_ as umap
except ImportError as e:
    raise ImportError("UMAP is not installed. Run `pip install umap-learn` and try again.") from e

# =========================
# Config
# =========================
CACHE_FILES = {
    "PlantVillage": Path("/root/DINO3/cache/dinov3_plantvillage_color_embeddings.pt"),
    "PlantDoc": Path("/root/DINO3/cache/dinov3_plantdoc_embeddings.pt"),
    "PlantWild": Path("/root/DINO3/cache/dinov3_plantwild_embeddings.pt"),
}

# 비교할 도메인 선택
ACTIVE_DOMAINS = ["PlantVillage", "PlantDoc", "PlantWild"]

# crop-wise cosine은 full 임베딩 기반, UMAP은 샘플링 기반
RANDOM_STATE = 42
MAX_SAMPLES_PER_DOMAIN = 4000

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "cosine"

TARGET_CROPS = [
    "apple",
    "bell pepper",
    "blueberry",
    "cherry",
    "corn",
    "grape",
    "peach",
    "potato",
    "raspberry",
    "soybean",
    "squash",
    "strawberry",
    "tomato",
]
TARGET_CROP_SET = set(TARGET_CROPS)

DOMAIN_COLORS = {
    "PlantVillage": "#1f77b4",
    "PlantDoc": "#ff7f0e",
    "PlantWild": "#2ca02c",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

PLANTVILLAGE_ROOT = Path("/root/DINO3/dataset/PlantVillage")
PLANTDOC_ROOT = Path("/root/DINO3/dataset/PlantDoc-Object-Detection-Dataset")
PLANTWILD_ROOT = Path("/root/DINO3/dataset/plantwild/images")


def extract_binary_label(domain, class_label):
    s = str(class_label).lower()

    # ------------------
    # PlantVillage
    # ------------------
    if domain == "PlantVillage":
        if "healthy" in s:
            return "healthy"
        return "disease"

    # ------------------
    # PlantDoc
    # ------------------
    if domain == "PlantDoc":

        healthy_keywords = [
            "leaf",
        ]

        disease_keywords = [
            "blight",
            "spot",
            "rust",
            "mold",
            "virus",
            "mite",
            "bacterial",
            "curl",
        ]

        for k in disease_keywords:
            if k in s:
                return "disease"

        return "healthy"

    # ------------------
    # PlantWild
    # ------------------
    if domain == "PlantWild":

        healthy_patterns = [
            " leaf"
        ]

        disease_keywords = [
            "blight",
            "spot",
            "rust",
            "virus",
            "mildew",
            "mosaic",
            "curl",
            "canker",
            "greening",
            "disease",
            "anthracnose",
            "smut",
        ]

        for k in disease_keywords:
            if k in s:
                return "disease"

        return "healthy"

    return "unknown"

def load_cached_embeddings(cache_path: Path) -> np.ndarray:
    if not cache_path.exists():
        raise FileNotFoundError(f"Cache not found: {cache_path}")

    try:
        payload = torch.load(cache_path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(cache_path, map_location="cpu")

    if not isinstance(payload, dict) or "embeddings" not in payload:
        raise ValueError(f"Invalid cache format: {cache_path}")

    emb = payload["embeddings"]
    if not torch.is_tensor(emb):
        raise ValueError(f"`embeddings` is not a torch tensor: {cache_path}")

    return emb.detach().cpu().numpy().astype(np.float32)


def sample_embeddings_with_indices(
    emb: np.ndarray,
    max_samples: int | None,
    rng: np.random.Generator,
):
    n = emb.shape[0]
    if (max_samples is None) or (n <= max_samples):
        idx = np.arange(n, dtype=np.int64)
        return emb, idx

    idx = np.sort(rng.choice(n, size=max_samples, replace=False))
    return emb[idx], idx


def normalize_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norm, eps, None)


def _unique_preserve_order(values):
    seen = set()
    out = []
    for v in values:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


def load_plantvillage_samples():
    split_files = [
        PLANTVILLAGE_ROOT / "splits" / "color_train.txt",
        PLANTVILLAGE_ROOT / "splits" / "color_test.txt",
    ]

    image_paths = []
    for split_file in split_files:
        if not split_file.exists():
            continue
        for line in split_file.read_text(encoding="utf-8").splitlines():
            rel = line.strip()
            if not rel:
                continue
            p = PLANTVILLAGE_ROOT / rel
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                image_paths.append(p)

    if len(image_paths) == 0:
        # fallback
        color_root = PLANTVILLAGE_ROOT / "raw" / "color"
        image_paths = [
            p for p in color_root.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ]

    image_paths = sorted(set(image_paths))
    class_labels = [p.parent.name for p in image_paths]
    return image_paths, class_labels


def _collect_plantdoc_split_samples(split_name: str, csv_path: Path):
    split_dir = PLANTDOC_ROOT / split_name
    if not csv_path.exists():
        raise ValueError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    grouped = df.groupby("filename", sort=False)["class"].apply(
        lambda s: _unique_preserve_order([str(v).strip() for v in s if pd.notna(v) and str(v).strip()])
    )

    image_paths = []
    class_labels = []

    for filename, classes in grouped.items():
        p = split_dir / str(filename)
        if not p.exists() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        if len(classes) == 0:
            continue

        # 한 이미지에 객체 클래스가 여러 개면 결합 문자열 사용
        label = " | ".join(classes)
        image_paths.append(p)
        class_labels.append(label)

    return image_paths, class_labels


def load_plantdoc_samples():
    tr_paths, tr_labels = _collect_plantdoc_split_samples(
        "TRAIN", PLANTDOC_ROOT / "train_labels.csv"
    )
    te_paths, te_labels = _collect_plantdoc_split_samples(
        "TEST", PLANTDOC_ROOT / "test_labels.csv"
    )

    image_paths = tr_paths + te_paths
    class_labels = tr_labels + te_labels
    return image_paths, class_labels


def load_plantwild_samples():
    image_paths = [
        p for p in PLANTWILD_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    image_paths = sorted(image_paths)
    class_labels = [p.parent.name for p in image_paths]
    return image_paths, class_labels


def load_domain_samples(domain: str):
    if domain == "PlantVillage":
        return load_plantvillage_samples()
    if domain == "PlantDoc":
        return load_plantdoc_samples()
    if domain == "PlantWild":
        return load_plantwild_samples()
    raise ValueError(f"Unsupported domain: {domain}")


def extract_crop_name(domain: str, class_label: str) -> str:
    s = str(class_label).strip()

    if domain == "PlantVillage":
        crop_raw = s.split("___")[0].lower().strip()
        crop_raw = crop_raw.replace("_", " ").replace(",", " ")
        crop_raw = crop_raw.replace("(", " ").replace(")", " ")
        crop_raw = " ".join(crop_raw.split())

        alias = {
            "pepper bell": "bell pepper",
            "cherry including sour": "cherry",
            "corn maize": "corn",
        }
        return alias.get(crop_raw, crop_raw)

    if domain == "PlantDoc":
        t = s.split("|")[0].strip().lower()
        if "bell_pepper" in t or "bell pepper" in t:
            return "bell pepper"
        if t.startswith("apple"):
            return "apple"
        if t.startswith("blueberry"):
            return "blueberry"
        if t.startswith("cherry"):
            return "cherry"
        if t.startswith("corn"):
            return "corn"
        if t.startswith("grape"):
            return "grape"
        if t.startswith("peach"):
            return "peach"
        if t.startswith("potato"):
            return "potato"
        if t.startswith("raspberry"):
            return "raspberry"
        if t.startswith("soyabean") or t.startswith("soybean"):
            return "soybean"
        if t.startswith("squash"):
            return "squash"
        if t.startswith("strawberry"):
            return "strawberry"
        if t.startswith("tomato"):
            return "tomato"
        return t.split()[0] if t else "unknown"

    # PlantWild
    if domain == "PlantWild":
        t = str(class_label).lower().strip()
        t = t.replace("_", " ")
        t = " ".join(t.split())

        if t.startswith("bell pepper"):
            return "bell pepper"
        if t.startswith("grapevine"):
            return "grape"

        return t.split()[0] if t else "unknown"

def compute_class_centroids(embeddings: np.ndarray, class_labels: np.ndarray):
    unique_classes = np.array(sorted(set(class_labels.tolist())))
    centroids = []
    for cls in unique_classes:
        centroids.append(embeddings[class_labels == cls].mean(axis=0))
    return unique_classes, np.asarray(centroids, dtype=np.float32)



In [ ]:
from sklearn.manifold import TSNE

# =========================
# Target config
# =========================
TARGET_CROP = "tomato"          # 예: "tomato", "potato", "apple"
TARGET_CLASS_KEYWORD = "" # 예: "healthy", "late blight", "early blight"
                               # crop만 볼 거면 None

MAX_SAMPLES_PER_DOMAIN_CROP = 1000

def build_target_domain_shift_data(
    compare_data,
    target_crop,
    target_class_keyword=None,
    max_samples_per_domain=1000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    X_list = []
    y_domain_list = []
    y_class_list = []
    y_crop_list = []

    selected_info = {}

    for domain in compare_data["active_domains"]:
        d = compare_data["domain_data"][domain]

        emb = d["emb_full"]
        class_labels = d["class_labels_full"]
        crop_labels = d["crop_labels_full"]

        mask = crop_labels == target_crop

        if target_class_keyword is not None:
            key = target_class_keyword.lower()
            mask = mask & np.array([
                key in str(c).lower().replace("_", " ")
                for c in class_labels
            ])

        idx = np.where(mask)[0]

        if len(idx) == 0:
            print(f"[WARN] {domain}: no samples for crop={target_crop}, class_keyword={target_class_keyword}")
            continue

        if len(idx) > max_samples_per_domain:
            idx = np.sort(rng.choice(idx, size=max_samples_per_domain, replace=False))

        X_list.append(emb[idx])
        y_domain_list.append(np.array([domain] * len(idx), dtype=object))
        y_class_list.append(class_labels[idx])
        y_crop_list.append(crop_labels[idx])

        selected_info[domain] = {
            "n": len(idx),
            "classes": sorted(set(class_labels[idx].tolist())),
        }

        print(f"{domain}: selected={len(idx):,}")
        for cls in selected_info[domain]["classes"][:20]:
            print(f"  - {cls}")
        if len(selected_info[domain]["classes"]) > 20:
            print(f"  ... {len(selected_info[domain]['classes']) - 20} more classes")

    if len(X_list) == 0:
        raise ValueError("No samples selected. Check TARGET_CROP or TARGET_CLASS_KEYWORD.")

    return {
        "X": np.concatenate(X_list, axis=0),
        "y_domain": np.concatenate(y_domain_list, axis=0),
        "y_class": np.concatenate(y_class_list, axis=0),
        "y_crop": np.concatenate(y_crop_list, axis=0),
        "selected_info": selected_info,
    }


target_data = build_target_domain_shift_data(
    compare_data=compare_data,
    target_crop=TARGET_CROP,
    target_class_keyword=TARGET_CLASS_KEYWORD,
    max_samples_per_domain=MAX_SAMPLES_PER_DOMAIN_CROP,
    random_state=RANDOM_STATE,
)

print("Target X:", target_data["X"].shape)

In [ ]:
# 1) 캐시 임베딩 로드 + 도메인별 라벨 정렬/샘플 준비
rng = np.random.default_rng(RANDOM_STATE)

domain_data = {}

for domain in ACTIVE_DOMAINS:
    emb_full = load_cached_embeddings(CACHE_FILES[domain])
    image_paths, class_labels = load_domain_samples(domain)

    if len(class_labels) != emb_full.shape[0]:
        raise ValueError(
            f"[{domain}] label count and embedding count mismatch: "
            f"labels={len(class_labels):,}, embeddings={emb_full.shape[0]:,}"
        )

    class_labels = np.asarray(class_labels, dtype=object)
    crop_labels = np.asarray(
        [extract_crop_name(domain, c) for c in class_labels],
        dtype=object,
    )
    binary_labels = np.asarray(
        [extract_binary_label(domain, c) for c in class_labels],
        dtype=object,
    )

    emb_sampled, sampled_idx = sample_embeddings_with_indices(
        emb_full,
        MAX_SAMPLES_PER_DOMAIN,
        rng,
    )

    class_labels_sampled = class_labels[sampled_idx]
    crop_labels_sampled = crop_labels[sampled_idx]
    binary_labels_sampled = binary_labels[sampled_idx]

    domain_data[domain] = {
        "emb_full": emb_full,
        "class_labels_full": class_labels,
        "crop_labels_full": crop_labels,
        "binary_labels_full": binary_labels,

        "emb_sampled": emb_sampled,
        "class_labels_sampled": class_labels_sampled,
        "crop_labels_sampled": crop_labels_sampled,
        "binary_labels_sampled": binary_labels_sampled,
    }

    print(
        f"{domain}: full={emb_full.shape[0]:,}, sampled={emb_sampled.shape[0]:,}, "
        f"classes={len(set(class_labels.tolist()))}, crops={len(set(crop_labels.tolist()))}"
    )

X = np.concatenate([domain_data[d]["emb_sampled"] for d in ACTIVE_DOMAINS], axis=0)
y_domain = np.concatenate([
    np.array([d] * domain_data[d]["emb_sampled"].shape[0], dtype=object)
    for d in ACTIVE_DOMAINS
])

compare_data = {
    "domain_data": domain_data,
    "X": X,
    "y_domain": y_domain,
    "active_domains": list(ACTIVE_DOMAINS),
}

print(f"Combined sampled points for UMAP: {X.shape[0]:,}")



In [ ]:
X_t = target_data["X"]
y_domain_t = target_data["y_domain"]

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=min(UMAP_N_NEIGHBORS, X_t.shape[0] - 1),
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
    low_memory=True,
)

coords_umap = reducer.fit_transform(X_t)

plt.figure(figsize=(10, 8))

for domain in sorted(set(y_domain_t.tolist())):
    mask = y_domain_t == domain
    plt.scatter(
        coords_umap[mask, 0],
        coords_umap[mask, 1],
        s=10,
        alpha=0.7,
        c=DOMAIN_COLORS.get(domain, None),
        label=f"{domain} (n={mask.sum():,})",
    )

title = f"UMAP Domain Shift: crop={TARGET_CROP}"
if TARGET_CLASS_KEYWORD is not None:
    title += f", class~{TARGET_CLASS_KEYWORD}"

plt.title(title)
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(markerscale=2, frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
X_t = target_data["X"]
y_domain_t = target_data["y_domain"]

perplexity = min(30, max(5, (X_t.shape[0] - 1) // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric="cosine",
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
)

coords_tsne = tsne.fit_transform(X_t)

plt.figure(figsize=(10, 8))

for domain in sorted(set(y_domain_t.tolist())):
    mask = y_domain_t == domain
    plt.scatter(
        coords_tsne[mask, 0],
        coords_tsne[mask, 1],
        s=10,
        alpha=0.7,
        c=DOMAIN_COLORS.get(domain, None),
        label=f"{domain} (n={mask.sum():,})",
    )

title = f"t-SNE Domain Shift: crop={TARGET_CROP}"
if TARGET_CLASS_KEYWORD is not None:
    title += f", class~{TARGET_CLASS_KEYWORD}"

plt.title(title)
plt.xlabel("t-SNE-1")
plt.ylabel("t-SNE-2")
plt.legend(markerscale=2, frameon=False)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# 3) Cross-domain Crop-wise Class Cosine Similarity
if "compare_data" not in globals():
    raise RuntimeError("`compare_data` not found. Run embedding-load cell first.")

active_domains = compare_data["active_domains"]
domain_data = compare_data["domain_data"]

print("Plotting cross-domain crop-wise class cosine similarity matrices...")

def get_crop_class_centroids(domain: str, crop: str):
    emb = domain_data[domain]["emb_full"]
    cls = domain_data[domain]["class_labels_full"]
    crp = domain_data[domain]["crop_labels_full"]

    mask = (crp == crop)
    if not np.any(mask):
        return None, None

    return compute_class_centroids(emb[mask], cls[mask])

for crop in TARGET_CROPS:
    domains_with_crop = [
        domain for domain in active_domains
        if np.any(domain_data[domain]["crop_labels_full"] == crop)
    ]

    if len(domains_with_crop) < 2:
        continue

    domain_pairs = [
        (domains_with_crop[i], domains_with_crop[j])
        for i in range(len(domains_with_crop))
        for j in range(i + 1, len(domains_with_crop))
    ]

    fig, axes = plt.subplots(1, len(domain_pairs), figsize=(7 * len(domain_pairs), 5.5))
    if len(domain_pairs) == 1:
        axes = [axes]

    plotted_any = False

    for ax, (left_domain, right_domain) in zip(axes, domain_pairs):
        left_classes, left_centroids = get_crop_class_centroids(left_domain, crop)
        right_classes, right_centroids = get_crop_class_centroids(right_domain, crop)

        if left_classes is None or right_classes is None:
            ax.axis("off")
            continue

        left_norm = normalize_rows(left_centroids)
        right_norm = normalize_rows(right_centroids)
        sim = left_norm @ right_norm.T

        im = ax.imshow(sim, vmin=-1, vmax=1, cmap="viridis", aspect="auto")
        x_labels = [x[:30] for x in right_classes.tolist()]
        y_labels = [x[:30] for x in left_classes.tolist()]
        ax.set_xticks(range(len(x_labels)))
        ax.set_yticks(range(len(y_labels)))
        ax.set_xticklabels(x_labels, rotation=90, fontsize=7)
        ax.set_yticklabels(y_labels, fontsize=7)
        ax.set_xlabel(right_domain)
        ax.set_ylabel(left_domain)
        ax.set_title(
            f"{left_domain} vs {right_domain} | {crop}\n"
            f"{len(y_labels)} x {len(x_labels)} classes"
        )

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plotted_any = True

    if plotted_any:
        fig.suptitle(f"Cross-Domain Crop-wise Class Cosine Similarity: {crop}", y=1.02)
        plt.tight_layout()
        plt.show()

print("Done.")
